
### SETUP Import Functions

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import *


**Read CSV Files from Source Volume**

In [0]:
schema = StructType([
    StructField("emp_id", IntegerType()),
    StructField("emp_name", StringType()),
    StructField("dept", StringType()),
    StructField("location", StringType()),
    StructField("update_ts", TimestampType())
])

In [0]:
df_src = spark.read.format('csv')\
    .options(header='true')\
    .schema(schema)\
    .load('/Volumes/rde_practice/raw_data/source/emp_src.csv')

In [0]:
df_src.display()


**Add row_hash, load timestamp for audit** 

In [0]:
df_src = df_src.withColumn('row_hash', unhex(sha2(concat_ws('|',col("emp_name"),col('dept'),col("location")),256)))\
               .withColumn('eff_end_date', lit('9999-12-31'))\
               .withColumn('load_ts', current_timestamp())
 

In [0]:
df_src.display()

In [0]:
df_src.createOrReplaceTempView('src_Emp')

In [0]:
%sql

SELECT * FROM src_Emp

In [0]:
%sql

MERGE INTO rde_practice.raw_data.dim_employee AS trg
USING (
   -- PART A: Actual source records (to handle brand new inserts and updates)
   SELECT 
     src.emp_id AS merge_key, 
     src.* FROM src_EMP AS src
   
   UNION ALL
   
   -- PART B: Records that need to be inserted as 'New Versions' 
   -- We set the merge_key to NULL so they fall into 'WHEN NOT MATCHED'
   SELECT 
     NULL AS merge_key, 
     src.*
   FROM src_EMP AS src
   JOIN rde_practice.raw_data.dim_employee AS trg 
     ON src.emp_id = trg.emp_id
   WHERE trg.is_current = 'Y' 
     AND src.row_hash <> trg.row_hash
) AS staged_updates
ON trg.emp_id = staged_updates.merge_key

-- 1️⃣ CLOSE the old version
WHEN MATCHED 
  AND trg.is_current = 'Y' 
  AND staged_updates.row_hash <> trg.row_hash
THEN UPDATE SET
    trg.is_current = 'N',
    trg.eff_end_dt = staged_updates.update_ts

-- 2️⃣ INSERT new records (New IDs AND the 'NULL' key versions from PART B)
WHEN NOT MATCHED THEN
  INSERT (
    emp_id,
    emp_name,
    dept,
    location,
    row_hash,
    eff_start_dt,
    eff_end_dt,
    is_current,
    load_ts
  )
  VALUES (
    staged_updates.emp_id,
    staged_updates.emp_name,
    staged_updates.dept,
    staged_updates.location,
    staged_updates.row_hash,
    staged_updates.update_ts,
    DATE('9999-12-31'),
    'Y',
    staged_updates.load_ts
  );

In [0]:
%sql

SELECT * FROM rde_practice.raw_data.dim_employee

In [0]:
%sql

select * from _sqldf